# RepLog AI CRM Data EDA

This notebook profiles the CRM data used by RepLog AI. The goal is to understand the source tables, identify data-quality constraints that matter for the product, and choose defensible synthetic meeting notes for the demo workflow.

The source CRM CSV files are treated as immutable. Any synthetic data should be limited to realistic meeting notes/transcripts and app-generated runtime records such as meeting logs, tasks, and audit events.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../data')
files = {
    'accounts': 'accounts.csv',
    'products': 'products.csv',
    'sales_pipeline': 'sales_pipeline.csv',
    'sales_teams': 'sales_teams.csv',
    'data_dictionary': 'data_dictionary.csv',
}
frames = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
accounts = frames['accounts']
products = frames['products']
pipeline = frames['sales_pipeline']
teams = frames['sales_teams']
summary = pd.DataFrame([
    {
        'table': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_cells': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
    }
    for name, df in frames.items()
])
print(summary.to_string(index=False))


          table  rows  columns  missing_cells  duplicate_rows
       accounts    85        7             70               0
       products     7        3              0               0
 sales_pipeline  8800        8           6103               0
    sales_teams    35        3              0               0
data_dictionary    21        3              0               0


## Source Table Inventory

The CRM source is compact but sufficient for a realistic MVP. The main operational table is `sales_pipeline` with **8,800 opportunities**. The other tables provide account, product, sales-team, and field-definition context.

There are **no duplicate rows** in the loaded source tables. The missing cells are concentrated in fields where missingness is expected for open pipeline records, especially `close_date` and `close_value`.


In [2]:
missing = pd.concat(
    [df.isna().sum().rename(name) for name, df in frames.items()], axis=1
).fillna('').astype(str)
unique = pd.concat(
    [df.nunique(dropna=False).rename(name) for name, df in frames.items()], axis=1
).fillna('').astype(str)
print('Missing values by field')
print(missing.to_string())
print('
Unique values by field')
print(unique.to_string())


Missing values by field
                 accounts products sales_pipeline sales_teams data_dictionary
account               0.0                  1425.0                            
sector                0.0                                                    
year_established      0.0                                                    
revenue               0.0                                                    
employees             0.0                                                    
office_location       0.0                                                    
subsidiary_of        70.0                                                    
product                        0.0            0.0                            
series                         0.0                                           
sales_price                    0.0                                           
opportunity_id                                0.0                            
sales_agent                             

## Completeness And Cardinality Findings

The accounts, products, sales teams, and data dictionary tables are complete for their core fields. The account table has **70 missing `subsidiary_of` values**, which is normal because most accounts are not subsidiaries.

The pipeline table has **1,425 missing account values**, **500 missing engage dates**, and **2,089 missing close dates/values**. The close-date and close-value gaps align with open opportunities. Missing account values are important for validation because they create records the AI should not blindly write against without a reliable match.


In [3]:
stages = pipeline['deal_stage'].value_counts().rename_axis('deal_stage').reset_index(name='count')
stages['share'] = (stages['count'] / len(pipeline)).map(lambda x: f'{x:.1%}')
open_pipeline = pipeline[pipeline['deal_stage'].isin(['Prospecting', 'Engaging'])]
open_with_account = open_pipeline[open_pipeline['account'].notna() & (open_pipeline['account'] != '')]
print(stages.to_string(index=False))
print(f'
Open opportunities: {len(open_pipeline):,}')
print(f'Open opportunities with account: {len(open_with_account):,}')
print(f'Open opportunities missing account: {len(open_pipeline) - len(open_with_account):,}')


 deal_stage  count share
        Won   4238 48.2%
       Lost   2473 28.1%
   Engaging   1589 18.1%
Prospecting    500  5.7%

Open opportunities: 2,089
Open opportunities with account: 664
Open opportunities missing account: 1,425


## Pipeline Stage Findings

The pipeline has four source stages: `Won`, `Lost`, `Engaging`, and `Prospecting`. Closed opportunities dominate the historical dataset: **4,238 won** and **2,473 lost** deals.

For the MVP, open opportunities are defined as `Prospecting` or `Engaging`. There are **2,089 open opportunities**, but only **664 have an account name**. This is useful for the product story: the validation agent must check whether an AI proposal can be tied to a real, sufficiently specific CRM opportunity before writeback.


In [4]:
source_accounts = set(accounts['account'].dropna())
source_products = set(products['product'].dropna())
source_agents = set(teams['sales_agent'].dropna())
pipe_accounts = set(pipeline['account'].dropna()) - {''}
pipe_products = set(pipeline['product'].dropna())
pipe_agents = set(pipeline['sales_agent'].dropna())
integrity = pd.DataFrame([
    {'check': 'Pipeline accounts not in accounts table', 'count': len(pipe_accounts - source_accounts), 'values': ', '.join(sorted(pipe_accounts - source_accounts)) or 'None'},
    {'check': 'Pipeline products not in products table', 'count': len(pipe_products - source_products), 'values': ', '.join(sorted(pipe_products - source_products)) or 'None'},
    {'check': 'Products not used in pipeline', 'count': len(source_products - pipe_products), 'values': ', '.join(sorted(source_products - pipe_products)) or 'None'},
    {'check': 'Pipeline agents not in sales team table', 'count': len(pipe_agents - source_agents), 'values': ', '.join(sorted(pipe_agents - source_agents)) or 'None'},
    {'check': 'Sales team agents not used in pipeline', 'count': len(source_agents - pipe_agents), 'values': ', '.join(sorted(source_agents - pipe_agents)) or 'None'},
])
print(integrity.to_string(index=False))


                                  check  count                                                                       values
Pipeline accounts not in accounts table      0                                                                         None
Pipeline products not in products table      1                                                                       GTXPro
          Products not used in pipeline      1                                                                      GTX Pro
Pipeline agents not in sales team table      0                                                                         None
 Sales team agents not used in pipeline      5 Carl Lin, Carol Thompson, Elizabeth Anderson, Mei-Mei Johns, Natalya Ivanova


## Referential Integrity Findings

All non-empty pipeline account names match the accounts table, and all pipeline sales agents match the sales-team table. This makes the dataset reliable enough for deterministic CRM validation.

The main reference-data issue is a product naming mismatch: the product catalog contains `GTX Pro`, while the pipeline uses `GTXPro`. This should not be hidden. The validation layer should normalize or alias the name while surfacing the mismatch as a data-quality warning.

Five sales-team agents do not appear in the pipeline. This is not a blocker; it simply means the sales-team table is broader than the historical opportunities in the sample data.


In [5]:
sector_counts = accounts['sector'].value_counts().rename_axis('sector').reset_index(name='accounts')
product_catalog = products.sort_values('sales_price')[['product', 'series', 'sales_price']]
print('Accounts by sector')
print(sector_counts.to_string(index=False))
print('
Product catalog')
print(product_catalog.to_string(index=False))


Accounts by sector
            sector  accounts
            retail        17
         technolgy        12
           medical        12
         marketing         8
           finance         8
          software         7
     entertainment         6
telecommunications         6
          services         5
        employment         4

Product catalog
       product series  sales_price
    MG Special     MG           55
     GTX Basic    GTX          550
GTX Plus Basic    GTX         1096
   MG Advanced     MG         3393
       GTX Pro    GTX         4821
  GTX Plus Pro    GTX         5482
       GTK 500    GTK        26768


## Account And Product Context

The account base spans **10 sectors**, with retail, technology, and medical accounts providing enough variety for realistic sales scenarios. The product catalog is small, with **7 products** across the GTX, MG, and GTK series.

This small product catalog is useful for the MVP because product validation can be strict and easy to explain. It also makes the `GTXPro`/`GTX Pro` mismatch visible without creating too much complexity.


In [6]:
selected_ids = ['7TI1WTV9', 'OT0GOR3I', 'OAMIMSUU', 'BJE7KCZ1']
demo_candidates = pipeline[pipeline['opportunity_id'].isin(selected_ids)][
    ['opportunity_id', 'account', 'product', 'sales_agent', 'deal_stage', 'engage_date']
].copy()
demo_candidates['synthetic_note_theme'] = [
    'Delivery timing and integration risk',
    'Budget approved and proposal follow-up',
    'Competitor objection and pricing follow-up',
    'Pilot next step with security review',
]
print(demo_candidates.to_string(index=False))


opportunity_id                      account      product        sales_agent  deal_stage engage_date                       synthetic_note_theme
      7TI1WTV9             Acme Corporation       GTXPro          Zane Levy    Engaging  2017-03-04       Delivery timing and integration risk
      OT0GOR3I                Bluth Company       GTXPro    Darcel Schlecht Prospecting         NaN     Budget approved and proposal follow-up
      BJE7KCZ1 Genco Pura Olive Oil Company       GTXPro   Gladys Colclough Prospecting         NaN Competitor objection and pricing follow-up
      OAMIMSUU                       Cheers GTX Plus Pro Jonathan Berthelot Prospecting         NaN       Pilot next step with security review


## Demo Record Selection

The selected demo opportunities are all real records from the pipeline and are still open. They cover recognizable accounts, valid sales agents, and products that exercise both the happy path and the product-name mismatch.

These records are good anchors for synthetic meeting notes because they let the system demonstrate realistic CRM behavior: identify the account, match the product, choose an open opportunity, flag risks or buying signals, suggest a stage update, and create a follow-up task after human approval.


In [7]:
runtime_tables = pd.DataFrame([
    {'table': 'meeting_logs', 'purpose': 'Approved meeting summaries and extracted context'},
    {'table': 'tasks', 'purpose': 'Approved follow-up actions generated from meeting notes'},
    {'table': 'audit_log', 'purpose': 'Approval/rejection history, proposal JSON, validation JSON, and applied changes'},
])
print(runtime_tables.to_string(index=False))


       table                                                                         purpose
meeting_logs                                Approved meeting summaries and extracted context
       tasks                         Approved follow-up actions generated from meeting notes
   audit_log Approval/rejection history, proposal JSON, validation JSON, and applied changes


## Synthetic Data Decision

The source CRM data should remain unchanged. It already provides the structured business records needed for validation: accounts, products, sales agents, and opportunities.

The missing data category is unstructured post-meeting context. There are no transcripts, meeting summaries, objections, buying signals, or follow-up tasks in the source files. For that reason, synthetic data should be limited to a small set of realistic meeting notes tied to real open opportunities.

This is defensible because it mirrors a common sales-operations reality: the CRM contains pipeline structure, while valuable meeting context is often trapped in notes, voice memos, messaging threads, or memory. RepLog AI is designed to convert that unstructured input into reviewed, auditable CRM updates.
